# 04 — Algorithm comparison: Ridge vs XGBoost vs TabPFN

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = next(p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents]
                    if (p / "analysis" / "lib").exists())
sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from joblib import Parallel, delayed

from analysis.lib.data import (load_label, restricted_pool,
                               build_demographics_matrix, identify_binary_targets)
from analysis.lib.pipelines import make_base_learner_xgboost, make_base_learner_tabpfn
from analysis.lib.stacking import process_stacking_fold
from analysis.lib.cv import pooled_cv, loco_cv
from analysis.lib.stats import nadeau_bengio_test, bh_fdr
from analysis.lib.plotting import VARIABLE_LABELS

DATA_DIR   = PROJECT_ROOT / "data"
LACUNA_DIR = DATA_DIR / "lacuna" / "METAVCI_PSCI_LACUNA" / "derivatives"
BACKEND_OUT = PROJECT_ROOT / "outputs" / "backend_comparison"
FIG_DIR     = BACKEND_OUT / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)

CLIN_PATH = DATA_DIR / "tabular" / "clinical_data.csv"
assert CLIN_PATH.exists(), f"missing {CLIN_PATH} — see data/README.md"

RIDGE_PATHS = {
    "pooled": PROJECT_ROOT / "outputs" / "predictive_modeling_core" / "pooled" / "stacking_scores.csv",
    "loco":   PROJECT_ROOT / "outputs" / "predictive_modeling_core" / "loco" / "stacking_scores.csv",
}
BACKENDS = {
    "xgboost": lambda n_demo, n_total: make_base_learner_xgboost(n_demo, n_total, gpu=False),
    "tabpfn":  lambda n_demo, n_total: make_base_learner_tabpfn(n_demo, n_total, device="cuda"),
}
N_JOBS = {"xgboost": -1, "tabpfn": 1}  # TabPFN shares one GPU

In [ ]:
clinical_df = pd.read_csv(CLIN_PATH, index_col=0)
acute_pool = restricted_pool(LACUNA_DIR, clinical_df, all_of=["acuteinfarct"])
clinical = clinical_df.loc[acute_pool]

lesion_acute = load_label(LACUNA_DIR, "acuteinfarct", "rd",  "damagepct").reindex(acute_pool)
flnm_acute   = load_label(LACUNA_DIR, "acuteinfarct", "fnm", "tmap").reindex(acute_pool)
slnm_acute   = load_label(LACUNA_DIR, "acuteinfarct", "snm", "disconnectionpct").reindex(acute_pool)

X_demo_pooled, _, _, _, _ = build_demographics_matrix(clinical, exclude_cohort=False)
X_demo_loco,   _, _, _, _ = build_demographics_matrix(clinical, exclude_cohort=True)
binary_targets = identify_binary_targets(clinical)

loco_splits_all = list(loco_cv(clinical["demographics-cohort"]))


def splits_for(valid_mask):
    valid_idx = np.where(valid_mask.values)[0]
    mapping = {orig: i for i, orig in enumerate(valid_idx)}
    out = []
    for train, test in loco_splits_all:
        tr = np.array([mapping[o] for o in train if o in mapping], dtype=int)
        te = np.array([mapping[o] for o in test if o in mapping], dtype=int)
        if len(te) > 0 and len(tr) > 0:
            out.append((tr, te))
    return out

In [ ]:
# Compute (skipped per backend/scheme when CSV already present)
for backend, factory in BACKENDS.items():
    for scheme in ("pooled", "loco"):
        out_dir = BACKEND_OUT / backend / scheme
        out_csv = out_dir / "stacking_scores.csv"
        if out_csv.exists():
            print(f"[skip] {backend}/{scheme}: {out_csv} exists")
            continue
        out_dir.mkdir(parents=True, exist_ok=True)
        X_demographics = X_demo_pooled if scheme == "pooled" else X_demo_loco
        rows = []
        for target in binary_targets:
            y_series = clinical[target]
            valid_mask = y_series.notna()
            y = y_series[valid_mask].values
            X_base = X_demographics[valid_mask.values]
            n_demo = X_base.shape[1]
            view_configs = [
                ("baseline",     X_base),
                ("lesion_acute", np.hstack([X_base, lesion_acute.loc[valid_mask].values])),
                ("flnm_acute",   np.hstack([X_base, flnm_acute.loc[valid_mask].values])),
                ("slnm_acute",   np.hstack([X_base, slnm_acute.loc[valid_mask].values])),
            ]
            splits = (list(pooled_cv(seed=42).split(X_base, y)) if scheme == "pooled"
                      else splits_for(valid_mask))
            print(f"--- {backend}/{scheme}: {target} (N={len(y)}, {len(splits)} folds) ---")
            fold_results = Parallel(n_jobs=N_JOBS[backend])(
                delayed(process_stacking_fold)(
                    i, train, test, view_configs, y, n_demo, learner_factory=factory)
                for i, (train, test) in enumerate(splits))
            for r in fold_results:
                rows.append({
                    "Target": target, "Fold": r["Fold"],
                    "AUPRC": r["AUPRC"], "AUPRC_train": r["AUPRC_train"],
                    "AUROC": r.get("AUROC", float("nan")),
                    "AUROC_train": r.get("AUROC_train", float("nan"))})
        pd.DataFrame(rows).to_csv(out_csv, index=False)
        print(f"saved {out_csv}")

In [ ]:
# Load all three backends per scheme
PATHS = {
    scheme: {
        "Ridge":   RIDGE_PATHS[scheme],
        "XGBoost": BACKEND_OUT / "xgboost" / scheme / "stacking_scores.csv",
        "TabPFN":  BACKEND_OUT / "tabpfn"  / scheme / "stacking_scores.csv",
    } for scheme in ("pooled", "loco")
}
CLASSIFIER_PALETTE = {"Ridge": "#9B7EBD", "XGBoost": "#1f9b3a", "TabPFN": "#d95f02"}
CLASSIFIER_ORDER = ["Ridge", "XGBoost", "TabPFN"]


def load_classifier(name, path):
    if not path.exists():
        print(f"  [missing] {path}")
        return None
    df = pd.read_csv(path)
    keep = ["Target", "Fold", "AUPRC"] + (["AUPRC_train"] if "AUPRC_train" in df.columns else [])
    df = df[keep]
    df["Classifier"] = name
    return df


LOADED = {}
for scheme, by_clf in PATHS.items():
    LOADED[scheme] = {name: df for name, path in by_clf.items()
                      if (df := load_classifier(name, path)) is not None}
    print(scheme, "loaded:", list(LOADED[scheme]))


def aggregate(loaded):
    rows, gap_rows = [], []
    for name, df in loaded.items():
        agg = df.groupby("Target")["AUPRC"].agg(["mean", "sem", "count"]).reset_index()
        agg["Classifier"] = name
        rows.append(agg)
        if "AUPRC_train" in df.columns:
            d = df.copy()
            d["gap"] = d["AUPRC_train"] - d["AUPRC"]
            g = d.groupby("Target")["gap"].agg(["mean", "sem", "count"]).reset_index()
            g["Classifier"] = name
            gap_rows.append(g)
    mean_df = pd.concat(rows, ignore_index=True) if rows else pd.DataFrame()
    gap_df = pd.concat(gap_rows, ignore_index=True) if gap_rows else pd.DataFrame()
    return mean_df, gap_df


MEAN_DFS, GAP_DFS = {}, {}
for scheme, loaded in LOADED.items():
    MEAN_DFS[scheme], GAP_DFS[scheme] = aggregate(loaded)

In [ ]:
# Paired NB tests vs Ridge and head-to-head
def paired_nb(focal_df, ref_df, label):
    rows = []
    for tgt in sorted(set(focal_df["Target"]).intersection(ref_df["Target"])):
        f = focal_df[focal_df["Target"] == tgt].set_index("Fold")["AUPRC"]
        r = ref_df[ref_df["Target"] == tgt].set_index("Fold")["AUPRC"]
        common = f.index.intersection(r.index)
        if len(common) < 2:
            continue
        out = nadeau_bengio_test(f.loc[common].values, r.loc[common].values,
                                 n_splits=10, test_size=0.1, alternative="greater")
        d = (f.loc[common] - r.loc[common]).dropna()
        rows.append({"Comparison": label, "Target": tgt,
                     "mean_delta": float(d.mean()), "n_paired": int(len(d)),
                     "t_stat": out["t_stat"], "p": out["p"]})
    df = pd.DataFrame(rows)
    if len(df):
        df["p_adj_BH"] = bh_fdr(df["p"].values)
    return df


def all_pairwise(loaded):
    pairs = {"xgb_vs_ridge":    ("XGBoost", "Ridge"),
             "tabpfn_vs_ridge": ("TabPFN", "Ridge"),
             "xgb_vs_tabpfn":   ("XGBoost", "TabPFN"),
             "tabpfn_vs_xgb":   ("TabPFN", "XGBoost")}
    return {k: (paired_nb(loaded[a], loaded[b], f"{a} > {b}")
                if {a, b} <= loaded.keys() else pd.DataFrame())
            for k, (a, b) in pairs.items()}


PAIRWISE = {scheme: all_pairwise(loaded) for scheme, loaded in LOADED.items()}
for scheme, parts in PAIRWISE.items():
    summary = pd.concat([d for d in parts.values() if not d.empty], ignore_index=True)
    summary.to_csv(BACKEND_OUT / f"nb_significance_{scheme}.csv", index=False)
    print(f"=== {scheme} ===")
    print(summary.round(4) if len(summary) else "(no comparisons)")

In [ ]:
# Figures: test AUPRC bars (+ optional train-test gap subplot)
def fdr_sig(nb_df, tgt):
    if nb_df is None or nb_df.empty:
        return False
    row = nb_df[nb_df["Target"] == tgt]
    if row.empty or pd.isna(row["p_adj_BH"].iloc[0]):
        return False
    return float(row["p_adj_BH"].iloc[0]) < 0.05 and float(row["mean_delta"].iloc[0]) > 0


def _draw_bars(ax, df, present, targets_present, sig_lookup=None):
    n_clf = len(present)
    group_w = 0.85
    band_w = group_w / n_clf
    offsets = np.linspace(-group_w/2 + band_w/2, group_w/2 - band_w/2, n_clf)
    for x_idx, tgt in enumerate(targets_present):
        for g_idx, clf in enumerate(present):
            cx = x_idx + offsets[g_idx]
            row = df[(df["Target"] == tgt) & (df["Classifier"] == clf)]
            if row.empty:
                continue
            mean = float(row["mean"].iloc[0])
            se = float(row["sem"].iloc[0])
            ax.bar(cx, mean, width=band_w*0.95, color=CLASSIFIER_PALETTE[clf],
                   edgecolor="black", linewidth=0.6,
                   label=clf if x_idx == 0 else None)
            if se > 0:
                ax.errorbar(cx, mean, yerr=se, color="black", linewidth=0.8,
                            capsize=3, zorder=4)
            if sig_lookup is not None:
                marks = sig_lookup(tgt, clf)
                if marks.get("vs_ridge"):
                    ax.scatter(cx, mean + se + 0.015, marker="^", s=70, color="black", zorder=5)
                if marks.get("vs_other_ml"):
                    ax.scatter(cx, mean + se + 0.038, marker="p", s=80, color="black", zorder=5)
    for x in range(1, len(targets_present)):
        ax.axvline(x - 0.5, color="gray", linewidth=0.3, alpha=0.4)
    ax.set_xticks(range(len(targets_present)))


def plot_panel(scheme, mean_df, gap_df, pairwise, loaded, out_path, include_gap=True):
    sns.set_theme(style="whitegrid", font_scale=1.3)
    present = [c for c in CLASSIFIER_ORDER if c in loaded]
    if not present or mean_df.empty:
        print(f"[{scheme}] nothing to plot — backends present: {present}")
        return
    targets_present = [t for t in VARIABLE_LABELS if t in mean_df["Target"].unique()]
    has_gap = not gap_df.empty and include_gap

    n_rows = 2 if has_gap else 1
    fig, axes = plt.subplots(
        n_rows, 1, figsize=(max(11, 1.7 * len(targets_present)), 6.5 + (3.5 if has_gap else 0)),
        sharex=True, gridspec_kw={"height_ratios": [1.0, 0.55] if has_gap else [1.0]})
    ax_top = axes[0] if has_gap else axes
    ax_bot = axes[1] if has_gap else None

    def sig_lookup(tgt, clf):
        if clf == "XGBoost":
            return {"vs_ridge": fdr_sig(pairwise.get("xgb_vs_ridge"), tgt),
                    "vs_other_ml": fdr_sig(pairwise.get("xgb_vs_tabpfn"), tgt)}
        if clf == "TabPFN":
            return {"vs_ridge": fdr_sig(pairwise.get("tabpfn_vs_ridge"), tgt),
                    "vs_other_ml": fdr_sig(pairwise.get("tabpfn_vs_xgb"), tgt)}
        return {}

    _draw_bars(ax_top, mean_df, present, targets_present, sig_lookup=sig_lookup)
    ax_top.set_ylabel("Test AUPRC\n(mean ± SE across folds)", fontsize=11, fontweight="bold")

    if has_gap:
        _draw_bars(ax_bot, gap_df, present, targets_present, sig_lookup=None)
        ax_bot.axhline(0, color="black", linewidth=0.7)
        ax_bot.set_ylabel("Train − Test AUPRC\n(overfitting gap)", fontsize=11, fontweight="bold")
        ax_bot.set_xlabel("")
        ax_bot.set_xticklabels([VARIABLE_LABELS[t] for t in targets_present],
                               fontsize=10, rotation=0)
    else:
        ax_top.set_xticklabels([VARIABLE_LABELS[t] for t in targets_present],
                               fontsize=10, rotation=0)
        ax_top.set_xlabel("")

    from matplotlib.lines import Line2D
    extra = [
        Line2D([0], [0], marker="^", color="w", markerfacecolor="black",
               markeredgecolor="black", markersize=9, linewidth=0,
               label=r"> Ridge ($p_{FDR}<0.05$, NB)"),
        Line2D([0], [0], marker="p", color="w", markerfacecolor="black",
               markeredgecolor="black", markersize=10, linewidth=0,
               label=r"> other ML method ($p_{FDR}<0.05$, NB)"),
    ]
    h, l = ax_top.get_legend_handles_labels()
    ax_top.legend(h + extra, l + [e.get_label() for e in extra],
                  bbox_to_anchor=(0.5, -0.10), loc="upper center", ncol=3,
                  title="Classifier / markers", frameon=True, fontsize=10)
    plt.tight_layout()
    fig.savefig(out_path, dpi=200, bbox_inches="tight")
    fig.savefig(out_path.with_suffix(".pdf"), bbox_inches="tight")
    print(f"[{scheme}] saved {out_path}")


for scheme in ("pooled", "loco"):
    plot_panel(scheme, MEAN_DFS[scheme], GAP_DFS[scheme], PAIRWISE[scheme],
               LOADED[scheme],
               FIG_DIR / f"ridge_vs_xgboost_vs_tabpfn_{scheme}.png", include_gap=False)
    plot_panel(scheme, MEAN_DFS[scheme], GAP_DFS[scheme], PAIRWISE[scheme],
               LOADED[scheme],
               FIG_DIR / f"ridge_vs_xgboost_vs_tabpfn_{scheme}_with_gap.png", include_gap=True)